# Step 5 — Direction Detection from VRS

Loads a VRS recording and estimates the sound direction using GCC-PHAT.

Change `VRS_FILE` below to test any recording.

## Configuration

**Change `BASE` below to match your machine before running anything else.**
All other paths are derived automatically.

In [ ]:
from pathlib import Path

# ── Set your base path here ───────────────────────────────────────────────
# Change this to where your esas_project folder is on your machine.
# Everything else is derived automatically.
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ─────────────────────────────────────────────────────────────────────────

RECORDINGS = BASE / 'recordings'
ESC50_DIR  = BASE / 'ESC-50'
PANNS_CKPT = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
MODELS_DIR = Path('models')  # saved inside esas_clean
RESULTS_DIR = Path('results')  # saved inside esas_clean
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f'BASE:       {BASE}')
print(f'Recordings: {RECORDINGS.exists()}')
print(f'ESC-50:     {ESC50_DIR.exists()}')
print(f'PANNs:      {PANNS_CKPT.exists()}')

In [ ]:
import math
import numpy as np

# Change this to any VRS file in your recordings folder
VRS_FILE = RECORDINGS / 'ofire0_1.vrs'

SR = 48_000
D  = 0.060   # mic baseline metres
C  = 343.0   # speed of sound m/s

print(f'Loading: {VRS_FILE}')
assert VRS_FILE.exists(), f'File not found: {VRS_FILE}'

In [ ]:
from projectaria_tools.core import data_provider as dp

provider = dp.create_vrs_data_provider(str(VRS_FILE))
audio_id = provider.get_stream_id_from_label('mic')
n        = provider.get_num_data(audio_id)
chunks   = []
for i in range(n):
    frame,_ = provider.get_audio_data_by_index(audio_id, i)
    try:    raw = np.array(frame.data, dtype=np.float32)
    except: raw = np.array(frame.audio_array, dtype=np.float32)
    if raw.ndim==1 and len(raw)%7==0:
        chunks.append(raw.reshape(7,-1))
audio = np.concatenate(chunks, axis=1)
print(f'Duration: {audio.shape[1]/SR:.1f}s   Channels: {audio.shape[0]}')

In [ ]:
# Find onset and extract direct-path window
mono  = audio[0]
fs    = 512
rms   = [np.sqrt(np.mean(mono[i:i+fs]**2)) for i in range(0,len(mono)-fs,fs)]
rms   = np.array(rms)
bg    = float(np.median(np.sort(rms)[:max(1,len(rms)//3)]))
onset = next((i*fs for i,r in enumerate(rms) if r>max(bg*5,0.005)), 0)
s = onset + int(SR*0.05)
e = s + int(SR*1.5)
window = audio[:, s:min(e, audio.shape[1])]
print(f'Onset: {onset/SR:.2f}s   Window: {s/SR:.2f}\u2013{min(e,audio.shape[1])/SR:.2f}s')

In [ ]:
def gcc_phat(s1, s2):
    n    = 2*int(2**math.ceil(math.log2(max(len(s1),len(s2)))))
    X1   = np.fft.rfft(s1, n=n)
    X2   = np.fft.rfft(s2, n=n)
    cc   = X1*np.conj(X2)
    gcc  = np.fft.irfft(cc/(np.abs(cc)+1e-10), n=n)
    ml   = int(SR*D/C)
    gh   = np.concatenate([gcc[-ml:], gcc[:ml+1]])
    pk   = int(np.argmax(gh)) - ml
    return math.degrees(math.asin(np.clip(pk/SR*C/D,-1,1)))

def label(a):
    return 'FAR LEFT' if a<-45 else 'LEFT' if a<-15 else 'FRONT' if a<15 else 'RIGHT' if a<45 else 'FAR RIGHT'

angles = [gcc_phat(window[c1].astype(np.float32), window[c2].astype(np.float32))
          for c1,c2 in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]]

print('Mic pair estimates:')
for (c1,c2),a in zip([(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)], angles):
    print(f'  Mic ({c1},{c2}): {a:+.1f}\u00b0  [{label(a)}]')

median = float(np.median(angles))
bar    = ['-']*41; bar[20]=':'; bar[max(0,min(40,int((median+90)/180*41)))]='|'
print(f'\nResult: {median:+.1f}\u00b0  [{label(median)}]')
print(f'[{"" .join(bar)}]')

In [ ]:
import numpy as np

# All frontal (0 degree) VRS files
frontal_files = [
    RECORDINGS / 'ofire0_1.vrs',
    RECORDINGS / 'ofire0_2.vrs', 
    RECORDINGS / 'ofire0_3.vrs',
    RECORDINGS / 'ophon0_1.vrs',
    RECORDINGS / 'ophon0_2.vrs',
    RECORDINGS / 'ophon0_3.vrs',
    RECORDINGS / 'kfire0_1.vrs',
    RECORDINGS / 'kfire0_2.vrs',
    RECORDINGS / 'kfire0_3.vrs',
    RECORDINGS / 'kphon0_1.vrs',
    RECORDINGS / 'kphon0_2.vrs',
    RECORDINGS / 'kphon0_3.vrs',
]

errors = []
for vrs in frontal_files:
    if not vrs.exists():
        print(f'Missing: {vrs.name}')
        continue
    try:
        provider = dp.create_vrs_data_provider(str(vrs))
        audio_id = provider.get_stream_id_from_label('mic')
        n        = provider.get_num_data(audio_id)
        chunks   = []
        for i in range(n):
            frame,_ = provider.get_audio_data_by_index(audio_id, i)
            try:    raw = np.array(frame.data, dtype=np.float32)
            except: raw = np.array(frame.audio_array, dtype=np.float32)
            if raw.ndim==1 and len(raw)%7==0:
                chunks.append(raw.reshape(7,-1))
        audio  = np.concatenate(chunks, axis=1)
        mono   = audio[0]
        rms    = [np.sqrt(np.mean(mono[i:i+512]**2)) for i in range(0,len(mono)-512,512)]
        rms    = np.array(rms)
        bg     = float(np.median(np.sort(rms)[:max(1,len(rms)//3)]))
        onset  = next((i*512 for i,r in enumerate(rms) if r>max(bg*5,0.005)), 0)
        s = onset + int(SR*0.05)
        e = s + int(SR*1.5)
        window = audio[:, s:min(e,audio.shape[1])]
        angles = [gcc_phat(window[c1].astype(np.float32), window[c2].astype(np.float32))
                  for c1,c2 in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]]
        median = float(np.median(angles))
        error  = abs(median - 0.0)  # ground truth is 0 degrees
        errors.append(error)
        print(f'  {vrs.name:<25} angle={median:+.1f}°  error={error:.1f}°')
    except Exception as ex:
        print(f'  ERROR {vrs.name}: {ex}')

errors = np.array(errors)
print(f'\nFrontal MAE: {errors.mean():.1f}°')
print(f'Std:         {errors.std():.1f}°')
print(f'All errors:  {list(errors.round(1))}')